# Tray lane allocation model for Q6

This notebook reads the `data/*.json` instances and builds the MILP model in Pyomo.


In [ ]:
import json
import math
import pandas as pd
from pathlib import Path
import pyomo.environ as pyo


## Load data


In [ ]:
with open(Path("data/50_trays.json")) as f:
    data = json.load(f)


In [ ]:
tray_types = {
    t["tray_type_name"]: t
    for t in data["tray_types"]
}

trays = data["trays"]

tray_lengths = {
    tray["id"]: tray_types[tray["tray_type"]]["length"]
    for tray in trays
}


## Basic sets and modelling parameter

We use at most `n` candidate lanes, where `n` is the number of trays.  
This is a standard modelling assumption, since in the worst case each tray can be placed in its own lane.

The lower bound is the total tray length divided by the lane capacity, rounded up.


In [ ]:
I = sorted(tray_lengths.keys())
J = list(range(len(I)))

lane_capacity = 2.0
total_tray_length = sum(tray_lengths.values())
lower_bound_lanes = max(1, math.ceil(total_tray_length / lane_capacity))
n = len(I)


## Build the Pyomo model


In [ ]:
tray_type_map = {
    tray["id"]: tray["tray_type"]
    for tray in trays
}

In [ ]:
def build_model():
    model = pyo.ConcreteModel(name="tray_lane_bin_packing")

    model.I = pyo.Set(initialize=I)
    model.J = pyo.Set(initialize=J)  # usually J = range(n)

    model.l = pyo.Param(model.I, initialize=tray_lengths, within=pyo.NonNegativeReals)

    model.x = pyo.Var(model.I, model.J, domain=pyo.Binary)
    model.y = pyo.Var(model.J, domain=pyo.Binary)
    model.L = pyo.Var(model.J, domain=pyo.NonNegativeReals)

    # Each tray must be assigned to exactly one lane
    model.assign_con = pyo.ConstraintList()
    for i in model.I:
        model.assign_con.add(
            sum(model.x[i, j] for j in model.J) == 1
        )

    # Lane length definition
    model.length_con = pyo.ConstraintList()
    for j in model.J:
        model.length_con.add(
            model.L[j] == sum(model.l[i] * model.x[i, j] for i in model.I)
        )

    # A tray can only be assigned to a used lane
    model.link_con = pyo.ConstraintList()
    for i in model.I:
        for j in model.J:
            model.link_con.add(
                model.x[i, j] <= model.y[j]
            )

    # Symmetry breaking: used lanes appear before unused lanes
    model.symmetry_con = pyo.ConstraintList()
    J_list = list(model.J)
    for j in range(len(J_list) - 1):
        model.symmetry_con.add(
            model.y[J_list[j]] >= model.y[J_list[j+1]]
        )
        
    # Hard capacity constraint: each lane must be <= 2m
    model.capacity_con = pyo.ConstraintList()
    for j in model.J:
        model.capacity_con.add(
            model.L[j] <= lane_capacity * model.y[j]
        )

    # A used lane must contain at least one tray
    model.nonempty_con = pyo.ConstraintList()
    for j in model.J:
        model.nonempty_con.add(
            model.y[j] <= sum(model.x[i, j] for i in model.I)
        )

    model.lower_bound_con = pyo.Constraint(
        expr=sum(model.y[j] for j in model.J) >= lower_bound_lanes
    )

    # Objective: minimise number of used lanes
    model.obj = pyo.Objective(
        expr=sum(model.y[j] for j in model.J),
        sense=pyo.minimize
    )

    return model

In [ ]:
model = build_model()

solver = pyo.SolverFactory("cbc")
#solver.options["seconds"] = 30
#solver.options["ratio"] = 0.05

results = solver.solve(model, tee=True)
termination = results.solver.termination_condition
status = results.solver.status

print(f"Solver status: {status}")
print(f"Termination condition: {termination}")

if termination != pyo.TerminationCondition.optimal:
    raise RuntimeError(f"CBC did not prove optimality: {termination}")

used_lanes = int(round(sum(pyo.value(model.y[j]) for j in model.J)))
used_lanes

In [ ]:
tray_type_map = {
    tray["id"]: tray["tray_type"]
    for tray in trays
}

rows = []

for i in model.I:
    for j in model.J:
        if pyo.value(model.x[i, j]) > 0.5:
            rows.append({
                "tray_id": int(i),
                "lane_id": int(j),
                "type": tray_type_map[int(i)],
                "length": float(pyo.value(model.l[i])),
            })

solution_df = pd.DataFrame(rows)
lane_map = {
    old_lane_id: new_lane_id
    for new_lane_id, old_lane_id in enumerate(sorted(solution_df["lane_id"].unique()))
}
solution_df["lane_id"] = solution_df["lane_id"].map(lane_map).astype(int)
solution_df = solution_df.sort_values(["lane_id", "tray_id"]).reset_index(drop=True)

In [ ]:
lane_summary = (
    solution_df
    .groupby(["lane_id", "type"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

for col in ["small", "medium", "large"]:
    if col not in lane_summary.columns:
        lane_summary[col] = 0

lane_lengths = (
    solution_df
    .groupby("lane_id")["length"]
    .sum()
    .reset_index(name="lane_length")
)

lane_summary = (
    lane_summary
    .merge(lane_lengths, on="lane_id")
    [["lane_id", "small", "medium", "large", "lane_length"]]
    .sort_values("lane_id")
    .reset_index(drop=True)
)

lane_summary